# Stage 1 CPU data preparation and validation
Run the cells in order on a **CPU Colab runtime**. This notebook executes the repository's `prepare_dataset.py` and `validate_data.py` directly, so their shared implementation stays authoritative. No copying/editing of `__file__`, argparse or script bodies is needed.

Select the intended Google account when mounting Drive. The Colab mount is separate from the Codex Drive connection. Outputs and caches go under `phase1`; the dataset manifest passes automatically to validation. A failed preparation cannot enable validation. This notebook does not run models or establish whole-stage completion.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/Research')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO_URL = 'https://github.com/Drazr/Research.git'
BRANCH = 'phase1'
STORAGE_SCOPE = 'persistent-drive'
DATASET_MANIFEST = None
PREPARATION_COMMIT = None


## 1. Mount the intended Drive and obtain phase1
Use a fresh clone or an existing clean `phase1` checkout. Existing checkouts are never reset or updated automatically; the selected HEAD is recorded below. For a non-Colab run, change both paths above and set STORAGE_SCOPE to `local-development`.


In [ ]:
try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab: configure an explicit local root and local-development scope.')
else:
    drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive mount is unavailable.')

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Repository destination is nonempty; choose an empty path.')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)

def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if git_output('remote', 'get-url', 'origin').removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected the Drazr/Research repository.')
if git_output('branch', '--show-current') != BRANCH:
    raise RuntimeError('Select the phase1 branch before continuing.')
if git_output('status', '--porcelain'):
    raise RuntimeError('Use a clean checkout so the recorded code commit matches the scripts.')
CODE_COMMIT = git_output('rev-parse', 'HEAD')
PHASE1_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
print('Code commit:', CODE_COMMIT)
print('Storage root:', PHASE1_ROOT)


## 2. Install CPU data requirements and define the script runner
The runner streams script output, checks its exit status and requires a passed JSON report. No GPU packages are installed.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-data.txt')], check=True)

COMMON_ARGS = [
    '--phase1-root', str(PHASE1_ROOT),
    '--code-commit', CODE_COMMIT,
    '--storage-scope', STORAGE_SCOPE,
]

def run_stage1_script(name, extra_args=()):
    if git_output('rev-parse', 'HEAD') != CODE_COMMIT or git_output('status', '--porcelain'):
        raise RuntimeError('Repository changed after setup; rerun setup from a clean checkout.')
    command = [sys.executable, str(REPO / 'scripts' / 'stage1' / name), *COMMON_ARGS, *extra_args]
    report = None
    with subprocess.Popen(command, cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict) and 'status' in candidate:
                report = candidate
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'{name} failed (exit {returncode}); fix the error before continuing.')
    if report is None or report.get('status') != 'passed':
        raise RuntimeError(f'{name} did not return a passed report.')
    return report


## 3. Prepare the dataset
Runs once for both models and all four languages. Rerunning uses the script's verified cache/resume behavior.


In [ ]:
# Clear a previous success before starting a new attempt.
DATASET_MANIFEST = None
PREPARATION_COMMIT = None
preparation_report = run_stage1_script('prepare_dataset.py')
candidate = Path(preparation_report['dataset_manifest']).resolve()
candidate.relative_to(PHASE1_ROOT.resolve())
if candidate.name != '_COMPLETE.json' or not candidate.is_file():
    raise RuntimeError('Preparation did not produce its completion manifest.')
DATASET_MANIFEST = candidate
PREPARATION_COMMIT = CODE_COMMIT
print('Dataset manifest:', DATASET_MANIFEST)


## 4. Validate the prepared dataset
Automatically consumes the successful preparation output. Checks the data and deterministic reload. Model/L4 and persistent synchronization checks remain separate prerequisites.


In [ ]:
if DATASET_MANIFEST is None or PREPARATION_COMMIT != CODE_COMMIT:
    raise RuntimeError('Run the preparation cell successfully for this code commit first.')
validation_report = run_stage1_script(
    'validate_data.py', ['--dataset-manifest', str(DATASET_MANIFEST)]
)
print('CPU preparation and validation passed.')
print('Dataset manifest for later model checks:', DATASET_MANIFEST)
print('Validation report:', validation_report['report'])
